In [ ]:
#| default_exp card

In [ ]:
#| export
from __future__ import annotations

import re

from fastermodels.eval import wilson

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

`render_card` writes the card from measured values only: every number in it comes from `meta`, there is no
default that could be mistaken for a measurement. A latency that was not measured is written
`non mesurée`, never `0`.

`check_card` reads a card back and returns what a reader should not have to trust: the phrases in
`FORBIDDEN`, and any speedup claim on a line that does not name both a device and a runtime — `2.3x faster`
says nothing, and so does `2.3x on CPU`; `2.3x on CPU with onnxruntime` says where it was measured.

In [ ]:
#| export
FORBIDDEN = ('lossless', 'un moteur int8', 'an int8 engine', 'state-of-the-art', 'sota', 'verified',
             'nan', 'ok=false', 'inconclusive', 'todo', 'tbd', 'xxx', 'placeholder')

_DEVICES = ('cpu', 'gpu', 'orin', '5090', 'jetson')
_RUNTIMES = ('tensorrt', 'onnxruntime', 'ort', 'openvino', 'pytorch', 'torchscript', 'eager')
_SPEEDUP = re.compile(r'\b\d+(?:\.\d+)?\s?[x×](?!\w)', re.I)


def _accuracy(k, n):
    "k/n in percent, with its Wilson 95 % interval"
    lo, hi = wilson(k, n)
    return f"{k}/{n} = {100 * k / n:.2f} % (Wilson 95 % [{100 * lo:.2f}, {100 * hi:.2f}])"


def render_card(
    meta: dict,  # name, base_model, license, datasets, tags, scope_line, recipe, reference, rows, latency, provenance
) -> str:
    "Render the model card: front matter, scope, recipe, accuracy table, size, latency and provenance"
    ref, latency = meta['reference'], meta.get('latency')
    out = ['---', 'library_name: fastermodels', f"license: {meta['license']}", f"base_model: {meta['base_model']}", 'datasets:']
    out += [f"  - {d}" for d in meta.get('datasets', [])]
    out += ['tags:'] + [f"  - {t}" for t in meta.get('tags', ['fasterai'])]
    out += ['---', '', f"# {meta['name']}", '', meta['scope_line'], '', '## Recipe', '']
    out += [f"- `{k}`: {v}" for k, v in (meta.get('recipe') or {}).items()]
    out += ['', '## Accuracy', '', f"Reference: **{ref['name']}** — {_accuracy(ref['k'], ref['n'])}", '',
            '| artifact | file | params | bytes | accuracy | delta (pt) | 95 % CI | McNemar p | agreement | agreement kind |',
            '|---|---|---|---|---|---|---|---|---|---|']
    out += [f"| {r['artifact']} | `{r['file']}` | {r['params']:,} | {r['bytes']:,} | {_accuracy(r['k'], r['n'])} "
            f"| {r['delta']:+.2f} | [{r['lo']:+.2f}, {r['hi']:+.2f}] | {r['p_mcnemar']:.4f} | {r['agreement']:.4f} "
            f"| {r['agreement_kind']} |" for r in meta.get('rows', [])]
    out += ['', '## Latency', '']
    if not latency: out += ['non mesurée']
    else:
        out += ['| device | runtime | precision | batch | median (ms) | runs |', '|---|---|---|---|---|---|']
        out += [f"| {r['device']} | {r['runtime']} | {r['precision']} | {r['batch']} | {r['median_ms']} | {r['n_runs']} |"
                for r in latency]
    out += ['', '## Provenance', ''] + [f"- {k}: `{v}`" for k, v in (meta.get('provenance') or {}).items()]
    return '\n'.join(out) + '\n'


def check_card(
    text: str,  # the card to read back
) -> list[str]:
    "Every forbidden phrase and every speedup claim without both a device and a runtime on its line; empty when the card is clean"
    found = [p for p in FORBIDDEN if re.search(rf"\b{re.escape(p)}\b", text, re.I)]
    for line in text.splitlines():
        claim, low = _SPEEDUP.search(line), line.lower()
        if claim and not (any(w in low for w in _DEVICES) and any(w in low for w in _RUNTIMES)):
            found.append(f"{claim.group().strip()} without both a device and a runtime on its line")
    return found

In [ ]:
show_doc(render_card)

In [ ]:
show_doc(check_card)

---

## Usage

```python
from fastermodels import render_card, check_card

meta = {
    'name': 'resnet18-pruned', 'base_model': 'torchvision/resnet18', 'license': 'bsd-3-clause',
    'datasets': ['frgfm/imagenette'], 'tags': ['fasterai', 'pruning'],
    'scope_line': 'Imagenette, n=3925, Wilson half-width about 1 pt; pipeline evidence, not a published claim.',
    'recipe': {'prune': 'ratio 0.3, local, round_to 8', 'recovery': '3 epochs'},
    'reference': {'name': 'resnet18 fine-tuned on Imagenette', 'k': 3700, 'n': 3925},
    'rows': [{'artifact': 'pruned FP32', 'file': 'model.safetensors', 'params': 8_900_000, 'bytes': 35_600_000,
              'k': 3680, 'n': 3925, 'delta': -0.51, 'lo': -1.2, 'hi': 0.2, 'p_mcnemar': 0.12,
              'agreement': 1.0, 'agreement_kind': 'same-precision'}],
    'latency': None,
    'provenance': {'fasterai': '0.4.0', 'fastermodels': '0.1.0', 'torch': '2.9.1', 'measured_on': '2026-09-11'},
}

card = render_card(meta)
check_card(card)   # [] — nothing a reader has to take on trust
```

The card is written to `README.md` in the artifact directory, which is also what the Hub shows.

---

## See Also

- [Eval](01_eval.html) - where `k`, `n`, the delta and the agreement come from
- [Gate](03_gate.html) - condition 7 refuses to publish a card `check_card` flags
- [Model](00_model.html) - the artifact the card describes

Tests live in `nbs/tests/test_card.ipynb`.